# EEG Analysis Pipeline: Single Subject

This notebook performs a full EEG analysis pipeline on a single subject, including parameter setup, data preprocessing, Granger causality computation, and results visualization.

In [4]:
# --- 1. Define Parameters and Setup ---
# Adjust these parameters for your analysis

data_path = './Dataset/Infants_data/sub-NORB00001/ses-1/eeg/sub-NORB00001_ses-1_task-EEG_eeg.edf'  # Path to EDF file
l_freq = 0.5      # Lower band-pass filter frequency (Hz)
h_freq = 30.0     # Upper band-pass filter frequency (Hz)
exclude_channels = ['T3', 'T4', 'T5', 'T6']  # Channels to drop (can be empty)
ica_components_to_exclude = [0, 1]           # ICA components to exclude (can be empty)
window_length_sec = 5.0                      # Window length in seconds
overlap_ratio = 0.4                          # Fractional overlap between windows
max_lag = 3                                  # Maximum lag for Granger causality test

# Import libraries
import mne
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import grangercausalitytests
import networkx as nx


In [ ]:
# --- 2. Data Loading, Preprocessing, and Artifact Removal ---
# Load EDF file
raw = mne.io.read_raw_edf(data_path, preload=True, verbose=False)
print(f"Loaded EDF: {len(raw.ch_names)} channels, {raw.times[-1]:.1f}s duration")

# Band-pass filter
raw.filter(l_freq, h_freq, fir_design='firwin', verbose=False)
print(f"Filtered data: {l_freq}-{h_freq} Hz")

# Drop specified channels
if exclude_channels:
    raw.drop_channels(exclude_channels)
    print(f"Dropped channels: {exclude_channels}")

# ICA for artifact removal
from mne.preprocessing import ICA
ica = ICA(n_components=0.99, random_state=42, max_iter='auto')
ica.fit(raw)
# Find EOG-related components automatically
try:
    ica_eog, scores = ica.find_bads_eog(raw)
    components_to_exclude = list(ica_eog)
    print(f"Auto-detected EOG components: {components_to_exclude}")
except RuntimeError:
    components_to_exclude = []
    print("No EOG channels found; skipping automatic EOG component exclusion.")
# Add manual exclusions
if ica_components_to_exclude:
    components_to_exclude += ica_components_to_exclude
if components_to_exclude:
    ica.exclude = components_to_exclude
    raw = ica.apply(raw)
    print(f"Excluded ICA components: {components_to_exclude}")
else:
    print("No ICA components excluded.")

# Common average reference
raw.set_eeg_reference('average', projection=True)
print("Applied common average reference.")

# Epoch data into non-overlapping segments (5s)
epoch_length = window_length_sec  # seconds
n_samples = int(epoch_length * raw.info['sfreq'])
start_indices = np.arange(0, raw.n_times, n_samples)
epochs = [raw.copy().crop(tmin=idx/raw.info['sfreq'], tmax=(idx+n_samples-1)/raw.info['sfreq']) for idx in start_indices if idx+n_samples <= raw.n_times]
print(f"Created {len(epochs)} non-overlapping epochs of {epoch_length}s each.")


Loaded EDF: 21 channels, 714.0s duration
Filtered data: 0.5-30.0 Hz
Dropped channels: ['T3', 'T4', 'T5', 'T6']
Fitting ICA to data using 17 channels (please be patient, this may take a while)
Selecting by explained variance: 13 components
Fitting ICA took 1.3s.


RuntimeError: No EOG channel(s) found

In [ ]:
# --- 3. Sliding Window Granger Causality Analysis ---
# Convert preprocessed data to numpy array
preprocessed_data = raw.get_data()  # shape: (n_channels, n_samples)
sampling_frequency = raw.info['sfreq']
n_channels, n_samples = preprocessed_data.shape

window_samples = int(window_length_sec * sampling_frequency)
step_samples = int(window_samples * (1 - overlap_ratio))

results = []  # List to store P-value matrices for each window
window_times = []  # Start time of each window (seconds)

for start in range(0, n_samples - window_samples + 1, step_samples):
    end = start + window_samples
    window_data = preprocessed_data[:, start:end]
    pval_matrix = np.ones((n_channels, n_channels))
    for i in range(n_channels):
        for j in range(n_channels):
            if i != j:
                test_data = np.vstack([window_data[i], window_data[j]]).T
                try:
                    gc_result = grangercausalitytests(test_data, max_lag, verbose=False)
                    pvals = [gc_result[lag][0]['ssr_ftest'][1] for lag in range(1, max_lag+1)]
                    pval_matrix[i, j] = min(pvals)
                except Exception:
                    pval_matrix[i, j] = 1.0
    results.append(pval_matrix)
    window_times.append(start / sampling_frequency)
    print(f"Processed window {len(results)}: samples {start}-{end}")

print(f"\nAnalysis complete. Number of windows: {len(results)}")
print(f"Shape of each P-value matrix: {results[0].shape if results else 'No results'}")


In [ ]:
# --- 4. Visualization of Results ---
# Average Causality Heatmap
avg_pval_matrix = np.mean(np.array(results), axis=0)
plt.figure(figsize=(10, 8))
sns.heatmap(avg_pval_matrix, xticklabels=raw.ch_names, yticklabels=raw.ch_names, cmap='viridis', annot=False)
plt.title('Average Granger Causality P-value Matrix')
plt.xlabel('Channel (causal)')
plt.ylabel('Channel (effect)')
plt.show()

# Time-Series Plot of Causality (example: Fp1 -> Fp2)
ch1 = 'Fp1'
ch2 = 'Fp2'
if ch1 in raw.ch_names and ch2 in raw.ch_names:
    i = raw.ch_names.index(ch1)
    j = raw.ch_names.index(ch2)
    pvals_over_time = [mat[i, j] for mat in results]
    plt.figure(figsize=(8, 4))
    plt.plot(window_times, pvals_over_time, marker='o')
    plt.title(f'Granger Causality P-value Over Time: {ch2} → {ch1}')
    plt.xlabel('Window Start Time (s)')
    plt.ylabel('P-value')
    plt.grid(True)
    plt.show()
else:
    print(f"Channels {ch1} and/or {ch2} not found in data.")

# Network Graph Visualization
significance_threshold = 0.05
G = nx.DiGraph()
for i, src in enumerate(raw.ch_names):
    for j, tgt in enumerate(raw.ch_names):
        if i != j and avg_pval_matrix[i, j] < significance_threshold:
            G.add_edge(src, tgt, weight=avg_pval_matrix[i, j])
plt.figure(figsize=(10, 10))
pos = nx.circular_layout(G)
nx.draw(G, pos, with_labels=True, node_color='skyblue', edge_color='r', node_size=1200, arrowsize=20)
plt.title('EEG Granger Causality Network (avg p < 0.05)')
plt.show()

# Topographical Map Visualization (average outgoing causality per channel)
avg_outgoing = np.mean(avg_pval_matrix < significance_threshold, axis=1)
montage = raw.get_montage() if raw.get_montage() is not None else mne.channels.make_standard_montage('standard_1020')
raw.set_montage(montage, on_missing='ignore')
info = raw.info
plt.figure(figsize=(8, 6))
mne.viz.plot_topomap(avg_outgoing, info, show=True, cmap='Reds', contours=0)
plt.title('Topomap: Average Outgoing Causality (p < 0.05)')
plt.show()
